Week 4-START

In [1]:
# Import libraries
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)

# Load the train and test sets saved in Week 3
PROCESSED_DIR = Path("../data/processed")
train = pd.read_csv(PROCESSED_DIR / "train.csv")
test = pd.read_csv(PROCESSED_DIR / "test.csv")
print("Train:", train.shape, "| Test:", test.shape)

Train: (1792, 29) | Test: (448, 29)


### 1. Remove duplicates
In train, we drop customers whose copies have different Response labels, then keep one copy of each customer. In test, we only remove exact copies.

In [2]:
# Columns that describe the customer (everything except ID and the target)
feature_columns = [c for c in train.columns if c not in ("ID", "Response")]

# Train: drop duplicate groups whose copies have different Response labels
key = train[feature_columns].apply(lambda s: s.map(str)).agg("|".join, axis=1)
labels_per_group = train.groupby(key)["Response"].transform("nunique")
print("Train rows dropped (conflicting labels):", (labels_per_group > 1).sum())
train = train[labels_per_group == 1]

# Train: keep one copy of each remaining customer
before = len(train)
train = train.drop_duplicates(subset=feature_columns).reset_index(drop=True)
print("Train rows dropped (duplicates):", before - len(train))

# Test: remove exact copies only (same features and same label)
before = len(test)
test = test.drop_duplicates(subset=feature_columns + ["Response"]).reset_index(drop=True)
print("Test rows dropped (duplicates):", before - len(test))

Train rows dropped (conflicting labels): 34
Train rows dropped (duplicates): 144
Test rows dropped (duplicates): 36


### 2. Clean the data
- Drop ID, Z_CostContact, and Z_Revenue because they don't help prediction.
- Group marital status into partnered or not, since Married and Together respond at about 10% while Single, Divorced, and Widow respond at about 22%. Absurd and YOLO become missing.
- Merge "2n Cycle" into Master.
- Treat birth years before 1920 and the $666,666 income as missing.

In [3]:
def clean(df):
    # Drop the identifier and the two constant columns
    df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])

    # Group marital status into partnered (1) or not (0); Absurd and YOLO become missing
    partnered_map = {"Married": 1, "Together": 1,
                     "Single": 0, "Divorced": 0, "Widow": 0, "Alone": 0}
    df["Is_Partnered"] = df["Marital_Status"].map(partnered_map)
    df = df.drop(columns="Marital_Status")

    # 2n Cycle is a master's level degree
    df["Education"] = df["Education"].replace({"2n Cycle": "Master"})

    # Birth years before 1920 are data entry errors
    df.loc[df["Year_Birth"] < 1920, "Year_Birth"] = np.nan

    # $666,666 is a placeholder, not a real income
    df.loc[df["Income"] == 666666, "Income"] = np.nan

    # Convert enrollment date from text to a date
    df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
    return df

train = clean(train)
test = clean(test)
print("Missing values in train after cleaning:")
print(train.isnull().sum()[train.isnull().sum() > 0])

Missing values in train after cleaning:
Year_Birth       2
Income          17
Is_Partnered     2
dtype: int64


### 3. Remove inconsistent rows (train only)
In Week 3 we found 5 rows with spending but zero purchases, and 3 rows where deal purchases exceed total purchases. These numbers contradict each other, so we drop them from train. Test is left unchanged.

In [4]:
spend_columns = ["MntWines", "MntFruits", "MntMeatProducts",
                 "MntFishProducts", "MntSweetProducts", "MntGoldProds"]
purchase_columns = ["NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases"]

total_spend = train[spend_columns].sum(axis=1)
total_purchases = train[purchase_columns].sum(axis=1)

# Rows where spending and purchase counts contradict each other
bad_rows = ((total_spend > 0) & (total_purchases == 0)) | \
           (train["NumDealsPurchases"] > total_purchases)
print("Inconsistent training rows dropped:", bad_rows.sum())
train = train[~bad_rows].reset_index(drop=True)

Inconsistent training rows dropped: 6


### 4. Fill missing values
We did this after all rows are dropped, so the fill values come from the final training set. Income and birth year are filled with the training median, and the partnered flag with the most common value.

In [5]:
# Fill values are learned from the training set only
income_median = train["Income"].median()
birth_median = train["Year_Birth"].median()
partnered_mode = train["Is_Partnered"].mode()[0]

for df in (train, test):
    df["Income"] = df["Income"].fillna(income_median)
    df["Year_Birth"] = df["Year_Birth"].fillna(birth_median)
    df["Is_Partnered"] = df["Is_Partnered"].fillna(partnered_mode).astype(int)

print("Missing values left in train:", train.isnull().sum().sum())
print("Missing values left in test:", test.isnull().sum().sum())

Missing values left in train: 0
Missing values left in test: 0


### 5. Create new features
Age, tenure, total spend, total purchases, and total prior campaigns accepted. In Week 3  prior acceptances and tenure showed as strong signals for Response. Year_Birth and Dt_Customer are dropped after Age and Tenure are created. A Children column is not created because Kidhome and Teenhome affect response differently.

In [7]:
campaign_columns = ["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3",
                    "AcceptedCmp4", "AcceptedCmp5"]

# Reference date = latest enrollment in train (same date used for test)
reference_date = train["Dt_Customer"].max()

def add_features(df):
    df["Age"] = reference_date.year - df["Year_Birth"]
    df["Tenure_Days"] = (reference_date - df["Dt_Customer"]).dt.days
    df["Total_Spend"] = df[spend_columns].sum(axis=1)
    df["Total_Purchases"] = df[purchase_columns].sum(axis=1)
    df["Total_Prev_Accepted"] = df[campaign_columns].sum(axis=1)
    # Year_Birth and Dt_Customer are replaced by Age and Tenure_Days
    return df.drop(columns=["Year_Birth", "Dt_Customer"])

train = add_features(train)
test = add_features(test)
train[["Age", "Tenure_Days", "Total_Spend", "Total_Purchases",
       "Total_Prev_Accepted"]].describe().round(1)

,Age,Tenure_Days,Total_Spend,Total_Purchases,Total_Prev_Accepted
count,1608.0,1608.0,1608.0,1608.0,1608.0
mean,45.3,352.5,622.9,12.7,0.3
std,11.7,201.4,609.6,7.2,0.7
min,18.0,0.0,8.0,1.0,0.0
25%,37.0,180.0,70.0,6.0,0.0
50%,44.0,352.0,411.0,12.0,0.0
75%,55.0,525.0,1079.2,19.0,0.0
max,74.0,699.0,2524.0,32.0,4.0


### 6. Log transform
In Week 3 we saw the spending columns, NumDealsPurchases, and NumCatalogPurchases are right skewed (skew above 1). A log transform reduces the effect of very large values without removing real high spenders. Income is not transformed: once the $666,666 placeholder is removed, its skew is only 0.29, and a log transform made it worse (-1.13).

In [8]:
# Rows where spending and purchase counts contradict each other
bad_rows = ((train["Total_Spend"] > 0) & (train["Total_Purchases"] == 0)) | \
           (train["NumDealsPurchases"] > train["Total_Purchases"])
print("Inconsistent training rows dropped:", bad_rows.sum())
train = train[~bad_rows].reset_index(drop=True)

Inconsistent training rows dropped: 0


### 7. One hot encode education
Each education level becomes its own 0/1 column, so the model doesn't assume equal gaps between degree levels. Basic is dropped and used as the baseline.

In [9]:
# Basic is dropped and becomes the baseline category
train = pd.get_dummies(train, columns=["Education"], prefix="Edu",
                       drop_first=True, dtype=int)
test = pd.get_dummies(test, columns=["Education"], prefix="Edu",
                      drop_first=True, dtype=int)

# Make sure test has exactly the same columns as train
test = test.reindex(columns=train.columns, fill_value=0)
print("Education columns:", [c for c in train.columns if c.startswith("Edu")])

Education columns: ['Edu_Graduation', 'Edu_Master', 'Edu_PhD']


### 8. Standardize
Numeric columns are scaled to mean 0 and standard deviation 1, so features in different units are comparable. 0/1 columns and the target are not scaled. The scaler is fit on train only.

In [10]:
# Binary (0/1) columns and the target are not scaled
binary_columns = [c for c in train.columns if set(train[c].unique()) <= {0, 1}]
scale_columns = [c for c in train.columns if c not in binary_columns]

scaler = StandardScaler()
train[scale_columns] = scaler.fit_transform(train[scale_columns])
test[scale_columns] = scaler.transform(test[scale_columns])

print("Scaled columns:", scale_columns)
train[scale_columns].describe().round(2)

Scaled columns: ['Income', 'Kidhome', 'Teenhome', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'Age', 'Tenure_Days', 'Total_Spend', 'Total_Purchases', 'Total_Prev_Accepted']


,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,Age,Tenure_Days,Total_Spend,Total_Purchases,Total_Prev_Accepted
count,1608.00,1608.00,1608.00,1608.00,1608.00,1608.00,1608.00,1608.00,1608.00,1608.00,1608.00,1608.00,1608.00,1608.00,1608.00,1608.00,1608.00,1608.00,1608.00,1608.00
mean,0.00,0.00,-0.00,0.00,-0.00,-0.00,0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00,0.00,0.00,-0.00,0.00,0.00,0.00,0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-2.35,-0.83,-0.92,-1.72,-0.91,-0.67,-0.75,-0.70,-0.67,-0.86,-1.22,-1.52,-0.91,-1.79,-2.24,-2.35,-1.75,-1.01,-1.62,-0.44
25%,-0.78,-0.83,-0.92,-0.86,-0.84,-0.62,-0.68,-0.65,-0.64,-0.68,-0.69,-0.78,-0.91,-0.87,-0.97,-0.72,-0.86,-0.91,-0.93,-0.44
50%,-0.00,-0.83,-0.92,0.00,-0.38,-0.45,-0.44,-0.48,-0.45,-0.35,-0.16,-0.04,-0.24,-0.25,0.30,-0.11,-0.00,-0.35,-0.09,-0.44
75%,0.78,1.01,0.91,0.90,0.60,0.20,0.29,0.24,0.18,0.24,0.37,0.70,0.42,0.67,0.73,0.83,0.86,0.75,0.88,-0.44
max,5.19,2.86,2.74,1.69,3.47,4.29,6.65,3.97,4.10,5.42,6.73,7.76,8.42,2.20,6.24,2.46,1.72,3.12,2.69,5.31
